# Ejercicio 3 - Consultas directas sobre archivos Parquet

Todas las consultas leen los Parquet de `data/raw` con `read_parquet` o `glob`,
**sin importarlos a una tabla**. El SQL proviene de `sql/01_exploracion_inicial.sql`;
cada celda muestra el objetivo, la fuente y la decision (3.8) antes del resultado.

In [1]:
from lab_utils import conectar, ejecutar
import matplotlib.pyplot as plt

con = conectar()

## 3.1. Cantidad de archivos disponibles

In [2]:
ejecutar(con, "01_exploracion_inicial.sql", "01_cantidad_archivos")

**Cantidad de archivos disponibles** (`01_exploracion_inicial.sql` - `01_cantidad_archivos`)

- **Objetivo:** Verificar cuantos archivos Parquet de 2026 existen para cada tipo de taxi.
- **Fuente:** data/raw/yellow/2026/*.parquet y data/raw/green/2026/*.parquet
- **Decision:** Comparar el conteo local con los meses publicados por la TLC; un mes aun no publicado no se considera faltante.

```sql
WITH archivos AS (
    SELECT
        file,
        CASE
            WHEN file LIKE '%/yellow/%' THEN 'yellow'
            WHEN file LIKE '%/green/%' THEN 'green'
            ELSE 'desconocido'
        END AS tipo_taxi,
        regexp_extract(file, '2026-(\d{2})\.parquet$', 1) AS mes
    FROM glob('data/raw/*/2026/*.parquet')
)
SELECT
    tipo_taxi,
    count(*) AS cantidad_archivos,
    min(mes) AS primer_mes,
    max(mes) AS ultimo_mes
FROM archivos
GROUP BY tipo_taxi
UNION ALL
SELECT
    'total' AS tipo_taxi,
    count(*) AS cantidad_archivos,
    min(mes) AS primer_mes,
    max(mes) AS ultimo_mes
FROM archivos
ORDER BY tipo_taxi;
```

,tipo_taxi,cantidad_archivos,primer_mes,ultimo_mes
0,green,8,01,08
1,total,16,01,08
2,yellow,8,01,08


Hay 8 archivos por tipo (enero a agosto de 2026), lo que coincide con los meses que la
TLC tenia publicados. La ausencia de septiembre a diciembre no es un error.

## 3.2. Cantidad de registros

In [3]:
ejecutar(con, "01_exploracion_inicial.sql", "02_cantidad_registros")

**Cantidad de registros disponibles** (`01_exploracion_inicial.sql` - `02_cantidad_registros`)

- **Objetivo:** Contar los viajes disponibles por tipo de taxi y en total.
- **Fuente:** Todos los archivos Parquet de taxis amarillos y verdes de 2026.
- **Decision:** Usar union_by_name porque los dos tipos de taxi no poseen exactamente el mismo esquema.

```sql
WITH viajes AS (
    SELECT
        CASE
            WHEN filename LIKE '%yellow%' THEN 'yellow'
            WHEN filename LIKE '%green%' THEN 'green'
            ELSE 'desconocido'
        END AS tipo_taxi
    FROM read_parquet(
        'data/raw/*/2026/*.parquet',
        filename = true,
        union_by_name = true
    )
)
SELECT
    coalesce(tipo_taxi, 'total') AS tipo_taxi,
    count(*) AS cantidad_registros
FROM viajes
GROUP BY GROUPING SETS ((tipo_taxi), ());
```

,tipo_taxi,cantidad_registros
0,yellow,29703355
1,total,30040469
2,green,337114


In [4]:
por_archivo = ejecutar(con, "01_exploracion_inicial.sql", "03_registros_por_archivo")
por_archivo

**Registros por archivo mensual** (`01_exploracion_inicial.sql` - `03_registros_por_archivo`)

- **Objetivo:** Confirmar que cada archivo puede leerse y conocer su aporte al total.
- **Fuente:** Todos los archivos Parquet de taxis amarillos y verdes de 2026.
- **Decision:** Mantener el nombre del archivo en los resultados para facilitar la deteccion de meses vacios o atipicos.

```sql
SELECT
    CASE
        WHEN filename LIKE '%yellow%' THEN 'yellow'
        WHEN filename LIKE '%green%' THEN 'green'
        ELSE 'desconocido'
    END AS tipo_taxi,
    regexp_extract(filename, '([^/]+)\.parquet$', 1) || '.parquet' AS archivo,
    count(*) AS cantidad_registros
FROM read_parquet(
    'data/raw/*/2026/*.parquet',
    filename = true,
    union_by_name = true
)
GROUP BY tipo_taxi, archivo
ORDER BY tipo_taxi, archivo;
```

,tipo_taxi,archivo,cantidad_registros
0,green,green_tripdata_2026-01.parquet,40272
1,green,green_tripdata_2026-02.parquet,37373
2,green,green_tripdata_2026-03.parquet,44208
3,green,green_tripdata_2026-04.parquet,44238
4,green,green_tripdata_2026-05.parquet,44921
5,green,green_tripdata_2026-06.parquet,44163
6,green,green_tripdata_2026-07.parquet,41252
7,green,green_tripdata_2026-08.parquet,40687
8,yellow,yellow_tripdata_2026-01.parquet,3724889
9,yellow,yellow_tripdata_2026-02.parquet,3399866


Mas del 98 % de los registros son de Yellow Taxi. Para comparar servicios conviene usar
promedios, porcentajes o indicadores por viaje y no cantidades absolutas. Ningun mes
esta vacio.

## 3.3 y 3.4. Columnas y tipos de datos

In [5]:
ejecutar(con, "01_exploracion_inicial.sql", "04_esquema_yellow")

**Columnas y tipos de datos de Yellow Taxi** (`01_exploracion_inicial.sql` - `04_esquema_yellow`)

- **Objetivo:** Identificar el esquema inferido por DuckDB para los archivos de taxis amarillos.
- **Fuente:** data/raw/yellow/2026/*.parquet
- **Decision:** Conservar los nombres originales y normalizar solamente en consultas que combinen tipos de taxi.

```sql
DESCRIBE
SELECT *
FROM read_parquet(
    'data/raw/yellow/2026/*.parquet',
    union_by_name = true
);
```

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [6]:
ejecutar(con, "01_exploracion_inicial.sql", "05_esquema_green")

**Columnas y tipos de datos de Green Taxi** (`01_exploracion_inicial.sql` - `05_esquema_green`)

- **Objetivo:** Identificar el esquema inferido por DuckDB para los archivos de taxis verdes.
- **Fuente:** data/raw/green/2026/*.parquet
- **Decision:** Usar lpep_pickup_datetime y lpep_dropoff_datetime para Green Taxi, en lugar de asumir los nombres de Yellow Taxi.

```sql
DESCRIBE
SELECT *
FROM read_parquet(
    'data/raw/green/2026/*.parquet',
    union_by_name = true
);
```

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


Los esquemas no coinciden: Yellow usa `tpep_*_datetime` y Green `lpep_*_datetime`;
Green agrega `ehail_fee` y `trip_type`, y Yellow `Airport_fee`. Por eso se usa
`union_by_name = true` al leerlos juntos. Las fechas son `TIMESTAMP`, los montos y
distancias `DOUBLE` y los identificadores `INTEGER`/`BIGINT`.

## 3.5. Muestra de registros

In [7]:
ejecutar(con, "01_exploracion_inicial.sql", "06_muestra_registros")

**Muestra de registros** (`01_exploracion_inicial.sql` - `06_muestra_registros`)

- **Objetivo:** Observar valores representativos de ambos tipos de taxi con columnas comparables.
- **Fuente:** Todos los archivos Parquet de taxis amarillos y verdes de 2026.
- **Decision:** Unificar los nombres de las fechas solo en la salida analitica y conservar los archivos originales sin modificaciones.

```sql
WITH muestra AS (
    SELECT
        'yellow' AS tipo_taxi,
        tpep_pickup_datetime AS pickup_datetime,
        tpep_dropoff_datetime AS dropoff_datetime,
        passenger_count,
        trip_distance,
        fare_amount,
        total_amount,
        payment_type
    FROM read_parquet('data/raw/yellow/2026/*.parquet', union_by_name = true)

    UNION ALL

    SELECT
        'green' AS tipo_taxi,
        lpep_pickup_datetime AS pickup_datetime,
        lpep_dropoff_datetime AS dropoff_datetime,
        passenger_count,
        trip_distance,
        fare_amount,
        total_amount,
        payment_type
    FROM read_parquet('data/raw/green/2026/*.parquet', union_by_name = true)
)
SELECT *
FROM muestra
WHERE pickup_datetime >= TIMESTAMP '2026-01-01'
  AND pickup_datetime < TIMESTAMP '2027-01-01'
QUALIFY row_number() OVER (
    PARTITION BY tipo_taxi
    ORDER BY pickup_datetime, dropoff_datetime
) <= 5
ORDER BY tipo_taxi, pickup_datetime;
```

,tipo_taxi,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,fare_amount,total_amount,payment_type
0,green,2026-01-01 00:03:27,2026-01-01 00:48:45,<NA>,23.58,2.90,57.95,<NA>
1,green,2026-01-01 00:04:26,2026-01-01 00:26:02,1,4.79,26.10,28.60,1
2,green,2026-01-01 00:08:25,2026-01-01 00:18:15,1,1.35,11.40,19.98,1
3,green,2026-01-01 00:08:56,2026-01-01 00:40:38,1,15.07,60.40,62.90,2
4,green,2026-01-01 00:09:37,2026-01-01 00:19:42,1,2.47,13.50,19.20,1
5,yellow,2026-01-01 00:00:00,2026-01-01 00:16:00,<NA>,4.38,22.39,32.57,0
6,yellow,2026-01-01 00:00:02,2026-01-01 01:34:24,2,2.80,51.30,57.05,4
7,yellow,2026-01-01 00:00:09,2026-01-01 00:11:24,1,3.70,18.40,39.15,1
8,yellow,2026-01-01 00:00:10,2026-01-01 00:25:57,3,2.93,22.60,32.35,1
9,yellow,2026-01-01 00:00:11,2026-01-01 00:13:33,<NA>,2.46,14.83,18.83,0


La muestra ya deja ver `passenger_count` nulo y codigos de pago iguales a cero: un
archivo legible no necesariamente tiene todas sus variables completas.

## 3.6. Problemas de calidad

In [8]:
ejecutar(con, "01_exploracion_inicial.sql", "07_calidad_datos")

**Revision inicial de calidad de datos** (`01_exploracion_inicial.sql` - `07_calidad_datos`)

- **Objetivo:** Cuantificar valores nulos, fechas inconsistentes y valores no plausibles antes del analisis exploratorio.
- **Fuente:** Todos los archivos Parquet de taxis amarillos y verdes de 2026.
- **Decision:** No eliminar automaticamente registros; documentar los problemas y aplicar filtros explicitos segun la pregunta analitica.

```sql
WITH viajes AS (
    SELECT
        CASE
            WHEN filename LIKE '%yellow%' THEN 'yellow'
            WHEN filename LIKE '%green%' THEN 'green'
            ELSE 'desconocido'
        END AS tipo_taxi,
        coalesce(tpep_pickup_datetime, lpep_pickup_datetime) AS pickup_datetime,
        coalesce(tpep_dropoff_datetime, lpep_dropoff_datetime) AS dropoff_datetime,
        passenger_count,
        trip_distance,
        fare_amount,
        total_amount
    FROM read_parquet(
        'data/raw/*/2026/*.parquet',
        filename = true,
        union_by_name = true
    )
)
SELECT
    tipo_taxi,
    count(*) AS registros,
    count_if(pickup_datetime IS NULL) AS pickup_nulo,
    count_if(dropoff_datetime IS NULL) AS dropoff_nulo,
    count_if(
        pickup_datetime < TIMESTAMP '2026-01-01'
        OR pickup_datetime >= TIMESTAMP '2027-01-01'
    ) AS pickup_fuera_2026,
    count_if(dropoff_datetime < pickup_datetime) AS duracion_negativa,
    count_if(passenger_count IS NULL) AS pasajeros_nulos,
    count_if(passenger_count <= 0) AS pasajeros_no_positivos,
    count_if(trip_distance IS NULL) AS distancia_nula,
    count_if(trip_distance <= 0) AS distancia_no_positiva,
    count_if(fare_amount < 0) AS tarifa_negativa,
    count_if(total_amount < 0) AS total_negativo
FROM viajes
GROUP BY tipo_taxi
ORDER BY tipo_taxi;
```

,tipo_taxi,registros,pickup_nulo,dropoff_nulo,pickup_fuera_2026,duracion_negativa,pasajeros_nulos,pasajeros_no_positivos,distancia_nula,distancia_no_positiva,tarifa_negativa,total_negativo
0,green,337114,0,0,14,5,48775,4527,0,12212,999,1023
1,yellow,29703355,0,0,17,10,7716688,91359,0,952231,157364,161835


Se observan fechas de inicio fuera de 2026, fechas de fin anteriores al inicio,
millones de `passenger_count` nulos, distancias no positivas y montos negativos. No
todos significan lo mismo (un total negativo puede ser un reembolso), por lo que **no
se eliminan registros automaticamente**: cada analisis posterior declara su filtro.

## 3.9. Que significa consultar directamente un Parquet

DuckDB usa el archivo como fuente de la consulta: interpreta su esquema y lee solo las
columnas y bloques necesarios, sin copiar los datos a una tabla. Como Parquet es
columnar y guarda metadatos (minimos/maximos por bloque), se reduce la lectura de disco,
no se duplica el almacenamiento y no hace falta cargar todo en memoria. Ademas, el
patron `data/raw/*/2026/*.parquet` incorpora un mes nuevo sin reconstruir nada.